# Transfer Learning — Clasificación de Frutas

En este taller aplicamos **aprendizaje por transferencia** para clasificar imágenes de frutas. En lugar de entrenar una CNN desde cero, partimos de un modelo preentrenado en ImageNet (ResNet-18) y lo adaptamos a nuestro problema específico.

Usaremos el dataset **Fruits 360**, disponible en Kaggle, que contiene imágenes de más de 130 tipos de frutas y verduras sobre fondo blanco.

> **Google Colab:** Este notebook está preparado para correr directamente en Colab. Las primeras celdas instalan dependencias y descargan el dataset automáticamente. Solo necesitas tu archivo `kaggle.json`.

## ⚙️ Setup — Entorno y dependencias

Esta celda detecta si estás en Google Colab y configura las rutas de trabajo.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    BASE_DIR = '/content'
    print("Entorno: Google Colab")
else:
    BASE_DIR = '.'
    print("Entorno: Local")

DATA_DIR  = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Directorio de datos: {DATA_DIR}")

In [ ]:
!pip install -q kagglehub torchsummary

## 💾 (Opcional) Montar Google Drive

Si montas tu Drive, el modelo entrenado se guardará de forma **persistente** y no se perderá al cerrar Colab. También puedes poner el dataset en Drive para no descargarlo cada vez.

Omite esta celda si prefieres trabajar en el almacenamiento temporal de Colab.

In [ ]:
USE_DRIVE = False  # Cambia a True si quieres guardar en Google Drive

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_DIR = '/content/drive/MyDrive/frutas_model'
    os.makedirs(MODEL_DIR, exist_ok=True)
    print(f"Modelos se guardarán en: {MODEL_DIR}")
else:
    print("Drive no montado. Los modelos se guardan en almacenamiento temporal.")

## 📦 Descarga del dataset — Fruits 360

Usamos `kagglehub` (la librería oficial de Kaggle). Al ejecutar la celda te pedirá tu **usuario** y **API key** de Kaggle.

In [ ]:
import kagglehub

# Descarga el dataset — pedirá usuario y API key la primera vez
path = kagglehub.dataset_download("moltean/fruits")
print("Dataset descargado en:", path)

# Buscar las carpetas Training y Test dentro del path descargado
TRAIN_DIR, TEST_DIR = None, None
for root, dirs, _ in os.walk(path):
    if 'Training' in dirs and 'Test' in dirs:
        TRAIN_DIR = os.path.join(root, 'Training')
        TEST_DIR  = os.path.join(root, 'Test')
        break

if TRAIN_DIR:
    print(f"Training : {TRAIN_DIR}")
    print(f"Test     : {TEST_DIR}")
    print(f"Clases   : {len(os.listdir(TRAIN_DIR))}")
else:
    print("[ERROR] No se encontraron las carpetas Training/Test. Estructura descargada:")
    for root, dirs, _ in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level < 4:
            print('  ' * level + os.path.basename(root) + '/')

## 🔧 Verificar GPU

En Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

El entrenamiento en CPU es posible pero mucho más lento.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Sin GPU — el entrenamiento será más lento.")
    print("En Colab: Entorno de ejecución → Cambiar tipo → GPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDispositivio activo: {device}")

## Preprocesamiento de imágenes

ResNet-18 fue entrenado con imágenes de ImageNet de 224×224 píxeles, normalizadas con la media y desviación estándar del dataset ImageNet. Debemos aplicar las mismas transformaciones a nuestras imágenes.

Para el entrenamiento añadimos **data augmentation** (aumentado de datos): volteos horizontales y rotaciones aleatorias. Esto ayuda al modelo a generalizar mejor sin necesidad de más datos.

In [ ]:
# Normalización estándar de ImageNet
imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)

# Transformaciones para entrenamiento (con augmentation)
train_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(100),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    imagenet_normalize
])

# Transformaciones para validación/prueba (sin augmentation)
test_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(100),
    transforms.ToTensor(),
    imagenet_normalize
])

train_dataset = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset  = torchvision.datasets.ImageFolder(TEST_DIR,  transform=test_transform)

NUM_CLASSES = len(train_dataset.classes)
print(f"Clases totales : {NUM_CLASSES}")
print(f"Imágenes train : {len(train_dataset)}")
print(f"Imágenes test  : {len(test_dataset)}")

## Visualización del dataset

Antes de entrenar, veamos algunas imágenes del dataset para entender con qué estamos trabajando.

In [ ]:
def desnormalizar(tensor):
    """Revierte la normalización de ImageNet para visualizar la imagen."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
indices = np.random.choice(len(train_dataset), 18, replace=False)

for ax, idx in zip(axes.flat, indices):
    img, label = train_dataset[idx]
    img_vis = desnormalizar(img).permute(1, 2, 0).numpy()
    ax.imshow(img_vis)
    ax.set_title(train_dataset.classes[label], fontsize=7)
    ax.axis('off')

plt.suptitle('Muestra del dataset Fruits 360', fontsize=13)
plt.tight_layout()
plt.show()

## DataLoaders

Creamos los `DataLoader` para iterar el dataset en mini-lotes durante el entrenamiento.

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2 if IN_COLAB else 0  # En Colab los workers paralelos funcionan bien

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

print(f"Mini-lotes de entrenamiento : {len(train_loader)}")
print(f"Mini-lotes de prueba        : {len(test_loader)}")

## Modelo preentrenado: ResNet-18

Usamos **ResNet-18**, un modelo más ligero que VGG-16 pero igualmente efectivo para transfer learning. Su arquitectura usa **bloques residuales** (conexiones skip) que facilitan el entrenamiento de redes profundas.

Para adaptarlo a nuestro dataset:
1. Reemplazamos la capa final `fc` (diseñada para 1000 clases de ImageNet) por una nueva con `NUM_CLASSES` salidas.
2. **Congelamos** todas las capas convolucionales para proteger los pesos preentrenados en la primera fase.

In [ ]:
from torchvision.models import ResNet18_Weights

# Cargar ResNet-18 con pesos preentrenados en ImageNet
resnet = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Reemplazar la capa final para adaptarla a nuestras clases
in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, NUM_CLASSES)

# Congelar todas las capas excepto la nueva capa final
for name, param in resnet.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

resnet = resnet.to(device)

total_params     = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"Parámetros totales     : {total_params:,}")
print(f"Parámetros entrenables : {trainable_params:,}")

## Funciones de entrenamiento y evaluación

In [ ]:
def evaluate(model, loader):
    """Calcula la precisión del modelo sobre un DataLoader."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3, optimizer=None):
    """Entrena el modelo e imprime métricas por época."""
    loss_fn = nn.CrossEntropyLoss()
    if optimizer is None:
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr
        )

    history = {'train_acc': [], 'val_acc': [], 'train_loss': []}

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

        train_acc = correct / total
        val_acc   = evaluate(model, test_loader)
        avg_loss  = total_loss / len(train_loader)

        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(avg_loss)

        print(f"Época {epoch+1}/{epochs} — "
              f"Loss: {avg_loss:.4f} | "
              f"Train acc: {train_acc:.4f} | "
              f"Val acc: {val_acc:.4f}")

    return history

## Fase 1 — Entrenamiento del clasificador (capas congeladas)

Primero entrenamos **solo la capa final** (`fc`) con las capas convolucionales congeladas. Esto es rápido y estabiliza los pesos del clasificador antes de descongelar el resto de la red.

In [ ]:
print("=== Fase 1: entrenando solo la capa final ===")
history_fase1 = train_model(resnet, train_loader, test_loader, epochs=5, lr=1e-3)

## Fase 2 — Ajuste fino (fine-tuning)

Ahora **descongelamos las últimas capas** de ResNet-18 (`layer4` y `fc`) y continuamos entrenando con una tasa de aprendizaje más baja. Esto permite que el modelo ajuste sus filtros de alto nivel a las características visuales de las frutas.

> **Importante:** Usamos una tasa de aprendizaje mucho menor para las capas preentrenadas (`1e-5`) que para el clasificador (`1e-3`), para no destruir los pesos aprendidos en ImageNet.

In [ ]:
# Descongelar layer4 (últimas capas convolucionales) y fc
for name, param in resnet.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"Parámetros entrenables tras descongelar layer4: {trainable_params:,}")

# Optimizer con tasas de aprendizaje diferenciadas
optimizer_ft = torch.optim.AdamW([
    {'params': [p for n, p in resnet.named_parameters() if 'layer4' in n], 'lr': 1e-5},
    {'params': resnet.fc.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)

print("\n=== Fase 2: fine-tuning de layer4 + clasificador ===")
history_fase2 = train_model(
    resnet, train_loader, test_loader,
    epochs=5, optimizer=optimizer_ft
)

## Visualización de resultados

In [ ]:
all_train_acc = history_fase1['train_acc'] + history_fase2['train_acc']
all_val_acc   = history_fase1['val_acc']   + history_fase2['val_acc']
all_loss      = history_fase1['train_loss']+ history_fase2['train_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_train_acc, label='Train')
ax1.plot(all_val_acc,   label='Validación')
ax1.axvline(x=4.5, color='gray', linestyle='--', label='Inicio fine-tuning')
ax1.set_title('Precisión por época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()
ax1.grid(True)

ax2.plot(all_loss, color='orange', label='Loss')
ax2.axvline(x=4.5, color='gray', linestyle='--', label='Inicio fine-tuning')
ax2.set_title('Loss de entrenamiento por época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\nPrecisión final en test: {all_val_acc[-1]:.4f} ({all_val_acc[-1]*100:.2f}%)")

## Predicciones sobre imágenes individuales

Visualicemos predicciones del modelo sobre imágenes del conjunto de prueba.

In [ ]:
resnet.eval()
images, labels = next(iter(test_loader))
images_dev = images[:12].to(device)

with torch.no_grad():
    outputs = resnet(images_dev)
    probs   = torch.softmax(outputs, dim=1)
    preds   = outputs.argmax(dim=1).cpu()

class_names = train_dataset.classes

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    img_vis = desnormalizar(images[i]).permute(1, 2, 0).numpy()
    real  = class_names[labels[i]]
    pred  = class_names[preds[i]]
    conf  = probs[i, preds[i]].item()
    color = 'green' if pred == real else 'red'
    ax.imshow(img_vis)
    ax.set_title(f"Real: {real}\nPred: {pred} ({conf:.0%})", fontsize=7, color=color)
    ax.axis('off')

plt.suptitle('Predicciones del modelo (verde=correcto, rojo=error)', fontsize=12)
plt.tight_layout()
plt.show()

## Guardar el modelo

In [ ]:
model_path = os.path.join(MODEL_DIR, 'frutas_resnet18.pth')
torch.save(resnet.state_dict(), model_path)
print(f"Modelo guardado en: {model_path}")

# En Colab también puedes descargarlo a tu computadora
if IN_COLAB and not USE_DRIVE:
    from google.colab import files
    files.download(model_path)
    print("Descarga iniciada en tu navegador.")

## Cargar el modelo guardado

Para retomar el trabajo sin reentrenar desde cero:

In [ ]:
from torchvision.models import ResNet18_Weights

resnet_cargado = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet_cargado.fc = nn.Linear(resnet_cargado.fc.in_features, NUM_CLASSES)
resnet_cargado.load_state_dict(torch.load(model_path, map_location=device))
resnet_cargado = resnet_cargado.to(device)
print("Modelo cargado correctamente.")

acc = evaluate(resnet_cargado, test_loader)
print(f"Precisión del modelo cargado: {acc:.4f}")

## Conclusión

En este taller aplicamos transfer learning con **ResNet-18** para clasificar frutas en 131 categorías:

- **Fase 1:** Entrenamos solo la capa de clasificación final, con las capas convolucionales congeladas. Esto es rápido y produce resultados razonables desde el inicio.
- **Fase 2 (fine-tuning):** Descongelamos las últimas capas convolucionales y continuamos entrenando con tasas de aprendizaje diferenciadas, mejorando la precisión.

Comparado con el enfoque de Gatos vs. Perros (2 clases), este es un problema más difícil (131 clases), por lo que esperamos una precisión algo más baja. Sin embargo, gracias al transfer learning podemos alcanzar buena precisión con relativamente pocos datos y tiempo de entrenamiento.